# 👗 Kleding Classifier v2 — Nauwkeurig & Bulk

Dit notebook gebruikt **wargoninnovation/wargon-clothing-classifier**, een Vision Transformer (ViT) getraind op 30.000+ kledingfoto's met 27 specifieke categorieën.

De 27 klassen worden automatisch samengevoegd naar jouw 6 hoofdcategorieën:

| Jouw categorie | Wat het model herkent |
|---|---|
| 👕 T-shirt / Top | t-shirt, shirt, blouse, top, hoodie, sweater, tank top... |
| 👖 Broek / Jeans | jeans, trousers, shorts, leggings, joggers... |
| 🧥 Jas / Vest | jacket, coat, blazer, cardigan, vest, windbreaker... |
| 👗 Jurk / Rok | dress, skirt, mini dress, maxi dress... |
| 👟 Schoenen | shoes, boots, sneakers, sandals, heels... |
| 👜 Accessoires | bag, backpack, belt, scarf, hat, jewelry... |
| ❓ Overige | geen kleding herkend |

## Stap 1 — Installeren & importeren

Eenmalig uitvoeren om het model te downloaden (~350 MB).

In [ ]:
# Installeer benodigde pakketten (verwijder # als je dit nog niet hebt)
# !pip install transformers torch pillow matplotlib

from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os
from pathlib import Path
from collections import defaultdict

print('✅ Imports geslaagd!')

## Stap 2 — Model laden

Het model wordt de eerste keer gedownload (~350 MB). Daarna staat het lokaal op je pc gecached.

In [ ]:
MODEL_NAAM = 'wargoninnovation/wargon-clothing-classifier'

print('⏳ Model laden... (eerste keer: ~350 MB download)')
processor = AutoImageProcessor.from_pretrained(MODEL_NAAM)
model     = AutoModelForImageClassification.from_pretrained(MODEL_NAAM)
model.eval()

# Bekijk alle beschikbare labels van het model
alle_labels = list(model.config.id2label.values())
print(f'\n✅ Model geladen! {len(alle_labels)} klassen beschikbaar:')
for i, label in enumerate(alle_labels):
    print(f'   {i:>2}. {label}')

## Stap 3 — Categorieën koppelen

⚠️ **Belangrijk:** Run Stap 2 eerst. Bekijk welke labels het model heeft, en pas eventueel de mapping hieronder aan.

In [ ]:
# Mapping: model-label (lowercase) → jouw categorie
# Pas deze lijst aan op basis van de labels die je in Stap 2 zag!
LABEL_MAPPING = {
    # 👕 T-shirt / Top
    't-shirt':      '👕 T-shirt / Top',
    'shirt':        '👕 T-shirt / Top',
    'blouse':       '👕 T-shirt / Top',
    'top':          '👕 T-shirt / Top',
    'hoodie':       '👕 T-shirt / Top',
    'sweater':      '👕 T-shirt / Top',
    'sweatshirt':   '👕 T-shirt / Top',
    'tank top':     '👕 T-shirt / Top',
    'polo':         '👕 T-shirt / Top',
    'pullover':     '👕 T-shirt / Top',
    'turtleneck':   '👕 T-shirt / Top',

    # 👖 Broek / Jeans
    'jeans':        '👖 Broek / Jeans',
    'trousers':     '👖 Broek / Jeans',
    'pants':        '👖 Broek / Jeans',
    'shorts':       '👖 Broek / Jeans',
    'leggings':     '👖 Broek / Jeans',
    'joggers':      '👖 Broek / Jeans',
    'sweatpants':   '👖 Broek / Jeans',
    'cargo pants':  '👖 Broek / Jeans',

    # 🧥 Jas / Vest
    'jacket':       '🧥 Jas / Vest',
    'coat':         '🧥 Jas / Vest',
    'blazer':       '🧥 Jas / Vest',
    'cardigan':     '🧥 Jas / Vest',
    'vest':         '🧥 Jas / Vest',
    'windbreaker':  '🧥 Jas / Vest',
    'parka':        '🧥 Jas / Vest',
    'overcoat':     '🧥 Jas / Vest',
    'anorak':       '🧥 Jas / Vest',

    # 👗 Jurk / Rok
    'dress':        '👗 Jurk / Rok',
    'skirt':        '👗 Jurk / Rok',
    'mini dress':   '👗 Jurk / Rok',
    'maxi dress':   '👗 Jurk / Rok',
    'midi dress':   '👗 Jurk / Rok',
    'jumpsuit':     '👗 Jurk / Rok',
    'romper':       '👗 Jurk / Rok',

    # 👟 Schoenen
    'shoes':        '👟 Schoenen',
    'boots':        '👟 Schoenen',
    'sneakers':     '👟 Schoenen',
    'sandals':      '👟 Schoenen',
    'heels':        '👟 Schoenen',
    'loafers':      '👟 Schoenen',
    'flats':        '👟 Schoenen',

    # 👜 Accessoires
    'bag':          '👜 Accessoires',
    'backpack':     '👜 Accessoires',
    'belt':         '👜 Accessoires',
    'scarf':        '👜 Accessoires',
    'hat':          '👜 Accessoires',
    'cap':          '👜 Accessoires',
    'jewelry':      '👜 Accessoires',
    'watch':        '👜 Accessoires',
    'sunglasses':   '👜 Accessoires',
    'gloves':       '👜 Accessoires',
    'wallet':       '👜 Accessoires',
    'purse':        '👜 Accessoires',
}

KLEUREN = {
    '👕 T-shirt / Top':  '#e8a87c',
    '👖 Broek / Jeans':  '#6b8cba',
    '🧥 Jas / Vest':     '#7daa7d',
    '👗 Jurk / Rok':     '#c47db5',
    '👟 Schoenen':       '#e07070',
    '👜 Accessoires':    '#d4a84b',
    '❓ Overige':        '#aaaaaa',
}

def model_label_naar_categorie(model_label: str) -> str:
    """Zet een model-label om naar een van jouw 6 categorieën."""
    label_lower = model_label.lower().strip()
    # Exacte match
    if label_lower in LABEL_MAPPING:
        return LABEL_MAPPING[label_lower]
    # Gedeeltelijke match (bv. 'blue jeans' → 'jeans')
    for sleutelwoord, categorie in LABEL_MAPPING.items():
        if sleutelwoord in label_lower:
            return categorie
    return '❓ Overige'

print('✅ Categorieën geconfigureerd!')
print(f'   {len(LABEL_MAPPING)} model-labels gekoppeld aan 6 categorieën')

## Stap 4 — Classificatiefunctie

In [ ]:
def classificeer(img: Image.Image):
    """
    Classificeert een kledingafbeelding met het gespecialiseerde model.
    Geeft terug: label (jouw categorie), model_label (origineel), kans, top5
    """
    inputs = processor(images=img.convert('RGB'), return_tensors='pt')
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = F.softmax(outputs.logits, dim=-1)[0]
    
    # Top-5 voorspellingen van het model
    top5_kansen, top5_indices = torch.topk(probs, min(5, len(probs)))
    top5 = [
        {
            'model_label': model.config.id2label[int(idx)],
            'categorie':   model_label_naar_categorie(model.config.id2label[int(idx)]),
            'kans':        float(k)
        }
        for k, idx in zip(top5_kansen, top5_indices)
    ]
    
    beste = top5[0]
    return {
        'label':       beste['categorie'],
        'model_label': beste['model_label'],
        'kans':        beste['kans'],
        'top5':        top5,
    }

print('✅ Classificatiefunctie klaar!')

## Stap 5 — Bulk classificatie

Zet je foto's in een map genaamd **`kleding`** naast dit notebook.

In [ ]:
# 🔧 Pas aan indien nodig
MAP = 'kleding'
EXTENSIES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

if not os.path.isdir(MAP):
    print(f'⚠️  Map "{MAP}" niet gevonden!')
    print(f'   Maak een map "{MAP}" aan naast dit notebook en zet daar je foto\'s in.')
else:
    bestanden = sorted([
        f for f in Path(MAP).iterdir()
        if f.suffix.lower() in EXTENSIES
    ])

    if not bestanden:
        print(f'⚠️  Geen afbeeldingen gevonden in "{MAP}"')
    else:
        print(f'📁 {len(bestanden)} afbeeldingen gevonden\n')
        resultaten = []

        for i, pad in enumerate(bestanden, 1):
            try:
                img = Image.open(pad)
                r = classificeer(img)
                resultaten.append({'pad': pad, 'img': img.copy(), **r})
                print(f'  [{i:>3}/{len(bestanden)}] {pad.name:35s} → {r["label"]:22s} '
                      f'(model: {r["model_label"]}, {r["kans"]*100:.0f}%)')
            except Exception as e:
                print(f'  ❌ {pad.name}: {e}')

        print(f'\n✅ Klaar! {len(resultaten)} afbeeldingen geclassificeerd.')

## Stap 6 — Resultaten per categorie bekijken

In [ ]:
if not resultaten:
    print('Geen resultaten — run Stap 5 eerst.')
else:
    groepen = defaultdict(list)
    for r in resultaten:
        groepen[r['label']].append(r)

    print(f'📊 Samenvatting ({len(resultaten)} foto\'s):')
    for label in list(KLEUREN.keys()):
        if label in groepen:
            print(f'   {label}: {len(groepen[label])}x')

    print('\n' + '='*60)

    for label in list(KLEUREN.keys()):
        if label not in groepen:
            continue
        items = groepen[label]
        kleur = KLEUREN[label]
        print(f'\n{label} — {len(items)} foto(\'s)')

        cols = min(4, len(items))
        rows = (len(items) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))

        if len(items) == 1:
            axes = [axes]
        elif rows == 1:
            axes = list(axes)
        else:
            axes = axes.flatten().tolist()

        for ax, item in zip(axes, items):
            ax.imshow(item['img'])
            ax.axis('off')
            ax.set_title(
                f"{item['pad'].name}\n"
                f"Model: {item['model_label']}\n"
                f"{item['kans']*100:.0f}% zekerheid",
                fontsize=8, color=kleur, fontweight='bold'
            )
            for spine in ax.spines.values():
                spine.set_edgecolor(kleur)
                spine.set_linewidth(3)
                spine.set_visible(True)

        for ax in axes[len(items):]:
            ax.set_visible(False)

        fig.suptitle(label, fontsize=14, fontweight='bold', color=kleur, y=1.01)
        plt.tight_layout()
        plt.show()

## Stap 7 — Fout geclassificeerd? Handmatig corrigeren

Als het model iets fout heeft ingedeeld, kun je hier handmatig de categorie aanpassen.

In [ ]:
# Pas de bestandsnaam en nieuwe categorie aan
BESTANDSNAAM   = 'mijn_foto.jpg'       # naam van het bestand in de kleding-map
NIEUWE_CATEGORIE = '👕 T-shirt / Top'  # kies uit de categorieën hieronder

# Beschikbare categorieën:
# '👕 T-shirt / Top'
# '👖 Broek / Jeans'
# '🧥 Jas / Vest'
# '👗 Jurk / Rok'
# '👟 Schoenen'
# '👜 Accessoires'

for r in resultaten:
    if r['pad'].name == BESTANDSNAAM:
        oud = r['label']
        r['label'] = NIEUWE_CATEGORIE
        print(f'✅ {BESTANDSNAAM}: {oud} → {NIEUWE_CATEGORIE}')
        break
else:
    print(f'⚠️  "{BESTANDSNAAM}" niet gevonden in resultaten.')

# Voer Stap 6 daarna opnieuw uit om het bijgewerkte overzicht te zien

---
## 💡 Tips

**Model herkent een label dat niet in LABEL_MAPPING staat?**  
Voeg het toe aan de mapping in Stap 3:
```python
'denim jacket': '🧥 Jas / Vest',
```

**Alles komt als 'Overige'?**  
Controleer in Stap 2 welke exacte labels het model gebruikt en pas de mapping in Stap 3 aan.

**Foto's met een persoon erin werken beter** dan losstaande kledingstukken op een witte achtergrond — het model is getraind op gedragen kleding.

**Volgende stap:** resultaten exporteren naar JSON voor gebruik in KledingHelper.